# Final PC-WMV vs MV GSM8K — persistent Drive run

Resumes the existing `results (1).jsonl` from the shared Drive folder and continuously syncs `results_final.jsonl` back to that folder.

In [ ]:
import os, sys, torch

print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Select a T4 GPU runtime before continuing.")
print("GPU:", torch.cuda.get_device_name(0))

PROJECT_DIR = "/content/adaptive-prefix-consistency"
if not os.path.isdir(PROJECT_DIR):
    alt = "/content/drive/MyDrive/adaptive-prefix-consistency"
    if os.path.isdir(alt):
        PROJECT_DIR = alt
    else:
        raise FileNotFoundError("Set PROJECT_DIR to your adaptive-prefix-consistency project.")

os.chdir(PROJECT_DIR)
print("Project:", os.getcwd())


In [ ]:
from google.colab import auth
auth.authenticate_user()

import google.auth
from google.auth.transport.requests import Request
from googleapiclient.discovery import build

SCOPES = ["https://www.googleapis.com/auth/drive"]
creds, _ = google.auth.default(scopes=SCOPES)
if not creds.valid and creds.expired and creds.refresh_token:
    creds.refresh(Request())

drive_service = build("drive", "v3", credentials=creds)
print("Drive authentication ready.")


In [ ]:
from pathlib import Path
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload

DRIVE_FOLDER_ID = "1_cplhPK8suNoHVHznA0EpwK-7kCgZyxt"
OUT_DIR = Path(PROJECT_DIR) / "outputs" / "gsm8k_qwen06b_final"
OUT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_CHECKPOINT = OUT_DIR / "results.jsonl"

def find_drive_file(name):
    q = f"'{DRIVE_FOLDER_ID}' in parents and name = '{name}' and trashed = false"
    return drive_service.files().list(q=q, fields="files(id,name)", pageSize=10).execute().get("files", [])

final_files = find_drive_file("results_final.jsonl")
old_files = find_drive_file("results (1).jsonl")
source = final_files[0] if final_files else (old_files[0] if old_files else None)

if source is None:
    raise FileNotFoundError("No results_final.jsonl or results (1).jsonl found in the shared folder.")

request = drive_service.files().get_media(fileId=source["id"])
with open(LOCAL_CHECKPOINT, "wb") as fh:
    downloader = MediaIoBaseDownload(fh, request)
    done = False
    while not done:
        _, done = downloader.next_chunk()

count = sum(1 for line in open(LOCAL_CHECKPOINT, encoding="utf-8") if line.strip())
print("Using:", source["name"])
print("Local checkpoint:", LOCAL_CHECKPOINT)
print("Completed records:", count)


In [ ]:
import threading

stop_sync = threading.Event()

def upload_checkpoint():
    if not LOCAL_CHECKPOINT.exists():
        return

    local_creds, _ = google.auth.default(scopes=SCOPES)
    if local_creds.expired and local_creds.refresh_token:
        local_creds.refresh(Request())

    service = build("drive", "v3", credentials=local_creds)
    q = f"'${DRIVE_FOLDER_ID}' in parents"
    q = f"'{DRIVE_FOLDER_ID}' in parents and name = 'results_final.jsonl' and trashed = false"
    existing = service.files().list(q=q, fields="files(id)", pageSize=10).execute().get("files", [])
    media = MediaFileUpload(str(LOCAL_CHECKPOINT), mimetype="application/json", resumable=False)

    if existing:
        service.files().update(fileId=existing[0]["id"], media_body=media).execute()
    else:
        service.files().create(
            body={"name":"results_final.jsonl", "parents":[DRIVE_FOLDER_ID]},
            media_body=media, fields="id"
        ).execute()

    print("[Drive sync] checkpoint updated", flush=True)

def sync_loop():
    last = None
    while not stop_sync.is_set():
        try:
            if LOCAL_CHECKPOINT.exists():
                m = LOCAL_CHECKPOINT.stat().st_mtime_ns
                if m != last:
                    upload_checkpoint()
                    last = m
        except Exception as e:
            print("[Drive sync warning]", e, flush=True)
        stop_sync.wait(10)

sync_thread = threading.Thread(target=sync_loop, daemon=True)
sync_thread.start()
print("Drive checkpoint sync is active.")


In [ ]:
from pathlib import Path
yaml_text = 'model_id: Qwen/Qwen3-0.6B\ndataset: gsm8k\nsplit: test\n\nnum_problems: 1000\nseed: 42\n\nn_samples: 16\nk_regen: 1\n\nmax_new_tokens: 512\ntemperature: 0.7\ntop_p: 0.95\ndo_sample: true\n\nfixed_tau: 0.75\nn_pilot: 4\npc_weight: cubic\n\ndtype: float16\ndevice_map: auto\nload_in_4bit: false\nenable_thinking: false\n\noutput_dir: outputs/gsm8k_qwen06b_final\n\nadaptive_tau:\n  easy_agreement: 0.85\n  hard_agreement: 0.40\n  tau_easy: 0.45\n  tau_medium: 0.75\n  tau_hard: 0.90\n'
Path("configs/gsm8k_qwen06b.yaml").write_text(yaml_text, encoding="utf-8")
print("Final YAML installed.")


In [ ]:
import subprocess, time

cmd = [
    sys.executable, "-u", "-m", "src.run_experiment",
    "--config", "configs/gsm8k_qwen06b.yaml",
    "--num-problems", "1000",
    "--n-samples", "16",
    "--checkpoint", str(LOCAL_CHECKPOINT),
]

print("Starting final experiment. Existing checkpoint will be resumed.", flush=True)
start = time.time()

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

try:
    for line in iter(process.stdout.readline, ""):
        if line:
            print(line, end="", flush=True)
    process.stdout.close()
    return_code = process.wait()
except KeyboardInterrupt:
    print("\nStopping experiment...", flush=True)
    process.terminate()
    try:
        process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        process.kill()
    raise

print(f"\nFinished with return code {return_code}. Runtime: {(time.time()-start)/60:.2f} minutes.", flush=True)

if return_code != 0:
    raise subprocess.CalledProcessError(return_code, cmd)


In [ ]:
stop_sync.set()
sync_thread.join(timeout=15)
upload_checkpoint()
print("Final checkpoint synchronized to Drive.")
